# 166. Pipeline Parallel：怎样实现 GPipe 与 1F1B 调度，并分析 bubble 和激活内存？

> **面试问题：微批流水线为什么有 bubble？1F1B 的 warmup、steady、cooldown 和更新屏障怎样设计？**

## 先给结论

Pipeline Parallel 要同时满足 stage 内互斥、forward 依赖、backward 反向依赖和权重版本一致。1F1B 不改变一次 flush 的理论气泡下界，却把 backward 提前，从而显著降低在途激活峰值；若允许不同权重版本并存，还会引入 weight staleness，需要另行论证。

## 推荐的回答主线

1. 把每个 `(stage, microbatch, F/B)` 建模为有依赖的操作，而不是只画一张理想时间图。
2. 分别生成 GPipe 和 flush-1F1B 的本地顺序，用离散事件调度验证无冲突、无越依赖。
3. 比较 makespan、利用率、bubble 与每 stage 最大存活激活，解释 microbatch 数的作用。
4. 补齐加权 loss、梯度累积、optimizer barrier、通信包身份、失败恢复和 profiling。

## 本 Notebook 的实现边界

模拟器假设每个 F/B 都耗时 1 tick、通信即时且 stage 同构，只用于验证依赖和相对内存。真实系统需加入不均衡层、P2P 通信、拓扑、重计算、虚拟 stage 和 kernel 时间。

## 一手资料

- [GPipe](https://arxiv.org/abs/1811.06965)
- [PipeDream](https://arxiv.org/abs/1806.03377)
- [Megatron pipeline parallelism](https://arxiv.org/abs/2104.04473)


In [ ]:
import hashlib  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
from collections import defaultdict  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。

# 用非平凡的 stage/microbatch 数覆盖 warmup、steady 与 cooldown。
STAGES, MICRO_BATCHES = 4, 8  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Op:  # 定义承载本节状态与行为的数据结构。
    stage: int  # 执行当前语句以推进本节示例。
    microbatch: int  # 执行当前语句以推进本节示例。
    kind: str  # "F" 或 "B"

assert STAGES > 1  # 用受控断言验证关键不变量。
assert MICRO_BATCHES >= STAGES  # 用受控断言验证关键不变量。
assert Op(0, 0, "F") != Op(0, 0, "B")  # 用受控断言验证关键不变量。


## 1. 依赖图：F 向右，B 向左，每个 B 还依赖自己的 F

调度合法性的最低条件是：stage s 的 F(m) 等待 s-1 的 F(m)；B(m) 等待 s+1 的 B(m)，且不能早于自身 F(m)。每个 stage 同一 tick 只能执行一个操作。


In [ ]:
def dependencies(op, stages):  # 定义本节可复用的核心函数。
    deps = set()  # 计算并保存当前步骤的中间状态。
    if op.kind == "F" and op.stage > 0:  # 按当前条件选择后续控制路径。
        deps.add(Op(op.stage - 1, op.microbatch, "F"))  # 执行当前语句以推进本节示例。
    if op.kind == "B":  # 按当前条件选择后续控制路径。
        deps.add(Op(op.stage, op.microbatch, "F"))  # 执行当前语句以推进本节示例。
        if op.stage < stages - 1:  # 按当前条件选择后续控制路径。
            deps.add(Op(op.stage + 1, op.microbatch, "B"))  # 执行当前语句以推进本节示例。
    return deps  # 返回当前分支计算出的结果。

# 首 stage 的 forward 无跨 stage 依赖，首 stage backward 等待下一 stage backward。
assert dependencies(Op(0, 0, "F"), STAGES) == set()  # 用受控断言验证关键不变量。
assert Op(1, 0, "B") in dependencies(Op(0, 0, "B"), STAGES)  # 用受控断言验证关键不变量。
assert Op(STAGES - 1, 0, "F") in dependencies(Op(STAGES - 1, 0, "B"), STAGES)  # 用受控断言验证关键不变量。


## 2. 通用离散调度器：只在依赖完成后弹出本地队首

各 stage 的本地顺序由策略给出；全局调度器每 tick 拍一张已完成集合快照，避免同 tick 结果被相邻 stage 零延迟消费。若长期无进展就说明本地顺序与依赖形成死锁。


In [ ]:
def run_schedule(local_orders, stages):  # 定义本节可复用的核心函数。
    queues = [list(order) for order in local_orders]  # 计算并保存当前步骤的中间状态。
    completed, timeline, tick = set(), [], 0  # 计算并保存当前步骤的中间状态。
    total = sum(map(len, queues))  # 计算并保存当前步骤的中间状态。
    while len(completed) < total:  # 在终止条件满足前持续推进状态。
        before_tick = set(completed)  # 计算并保存当前步骤的中间状态。
        running = []  # 计算并保存当前步骤的中间状态。
        for stage in range(stages):  # 遍历输入元素以累积或检查结果。
            if queues[stage] and dependencies(queues[stage][0], stages) <= before_tick:  # 按当前条件选择后续控制路径。
                running.append(queues[stage].pop(0))  # 执行当前语句以推进本节示例。
        if not running:  # 按当前条件选择后续控制路径。
            raise RuntimeError("调度死锁")  # 遇到非法合同立即显式失败。
        completed.update(running)  # 执行当前语句以推进本节示例。
        timeline.append(running)  # 执行当前语句以推进本节示例。
        tick += 1  # 计算并保存当前步骤的中间状态。
    return timeline  # 返回当前分支计算出的结果。

# 一个 stage 的单微批顺序可以正常完成，且 F 必定在 B 之前。
tiny = run_schedule([[Op(0, 0, "F"), Op(0, 0, "B")]], 1)  # 计算并保存当前步骤的中间状态。
assert len(tiny) == 2  # 用受控断言验证关键不变量。
assert tiny[0][0].kind == "F"  # 用受控断言验证关键不变量。
assert tiny[1][0].kind == "B"  # 用受控断言验证关键不变量。


## 3. GPipe：所有 forward 完成后再 backward

GPipe 的 flush 顺序简单：每个 stage 先执行全部微批 forward，再执行 backward。这里 backward 按微批正序只是合法选择；训练实现也常按反序组织，关键是依赖和梯度归并语义一致。


In [ ]:
def gpipe_orders(stages, microbatches):  # 定义本节可复用的核心函数。
    return [  # 返回当前分支计算出的结果。
        [Op(s, m, "F") for m in range(microbatches)]  # 执行当前语句以推进本节示例。
        + [Op(s, m, "B") for m in range(microbatches)]  # 执行当前语句以推进本节示例。
        for s in range(stages)  # 遍历输入元素以累积或检查结果。
    ]  # 执行当前语句以推进本节示例。

# 每个 stage 恰好有 M 个 F 和 M 个 B，完整模拟不死锁。
gpipe_timeline = run_schedule(gpipe_orders(STAGES, MICRO_BATCHES), STAGES)  # 计算并保存当前步骤的中间状态。
gpipe_ops = [op for tick in gpipe_timeline for op in tick]  # 计算并保存当前步骤的中间状态。
assert len(gpipe_ops) == 2 * STAGES * MICRO_BATCHES  # 用受控断言验证关键不变量。
assert sum(op.kind == "F" for op in gpipe_ops) == STAGES * MICRO_BATCHES  # 用受控断言验证关键不变量。
assert len(gpipe_timeline) > 2 * MICRO_BATCHES  # 用受控断言验证关键不变量。


## 4. 1F1B：warmup 后交替，再 cooldown

flush-1F1B 在 stage s 先做 `p-s-1` 个 warmup forward（受 M 截断），之后交替发一个新 F 与处理一个旧 B，最后清空剩余 B。所有微批仍使用同一权重版本，optimizer 在 flush 末尾更新。


In [ ]:
def one_f_one_b_orders(stages, microbatches):  # 定义本节可复用的核心函数。
    orders = []  # 计算并保存当前步骤的中间状态。
    for s in range(stages):  # 遍历输入元素以累积或检查结果。
        warmup = min(stages - s - 1, microbatches)  # 计算并保存当前步骤的中间状态。
        order = [Op(s, m, "F") for m in range(warmup)]  # 计算并保存当前步骤的中间状态。
        for i in range(microbatches - warmup):  # 遍历输入元素以累积或检查结果。
            order.extend([Op(s, warmup + i, "F"), Op(s, i, "B")])  # 执行当前语句以推进本节示例。
        order.extend(Op(s, m, "B") for m in range(microbatches - warmup, microbatches))  # 执行当前语句以推进本节示例。
        orders.append(order)  # 执行当前语句以推进本节示例。
    return orders  # 返回当前分支计算出的结果。

# 1F1B 也完整执行所有操作，且第一 stage 的 warmup 数符合公式。
onef_orders = one_f_one_b_orders(STAGES, MICRO_BATCHES)  # 计算并保存当前步骤的中间状态。
onef_timeline = run_schedule(onef_orders, STAGES)  # 计算并保存当前步骤的中间状态。
onef_ops = [op for tick in onef_timeline for op in tick]  # 计算并保存当前步骤的中间状态。
assert len(onef_ops) == 2 * STAGES * MICRO_BATCHES  # 用受控断言验证关键不变量。
assert onef_orders[0][:STAGES - 1] == [Op(0, m, "F") for m in range(STAGES - 1)]  # 用受控断言验证关键不变量。
assert len({op for op in onef_ops}) == 2 * STAGES * MICRO_BATCHES  # 用受控断言验证关键不变量。


## 5. 自动验依赖、互斥与利用率，而不是凭甘特图目测

把每个操作映射到 tick，即可程序化检查所有依赖严格早于消费者、每 stage 每 tick 至多一个操作。利用率是有效 stage-ticks 除以 `makespan*stages`；同构简化下两种 flush 的 makespan 可能相近。


In [ ]:
def validate_timeline(timeline, stages):  # 定义本节可复用的核心函数。
    time_of = {op: tick for tick, running in enumerate(timeline) for op in running}  # 计算并保存当前步骤的中间状态。
    for tick, running in enumerate(timeline):  # 遍历输入元素以累积或检查结果。
        assert len({op.stage for op in running}) == len(running)  # 用受控断言验证关键不变量。
        for op in running:  # 遍历输入元素以累积或检查结果。
            assert all(time_of[dep] < tick for dep in dependencies(op, stages))  # 用受控断言验证关键不变量。
    utilization = len(time_of) / (len(timeline) * stages)  # 计算并保存当前步骤的中间状态。
    return utilization, time_of  # 返回当前分支计算出的结果。

# 两条策略均合法，利用率必须在 (0,1]，并包含不可消除的空槽。
gpipe_util, gpipe_time = validate_timeline(gpipe_timeline, STAGES)  # 计算并保存当前步骤的中间状态。
onef_util, onef_time = validate_timeline(onef_timeline, STAGES)  # 计算并保存当前步骤的中间状态。
assert 0 < gpipe_util <= 1 and 0 < onef_util <= 1  # 用受控断言验证关键不变量。
assert gpipe_util < 1.0  # 用受控断言验证关键不变量。
assert onef_util < 1.0  # 用受控断言验证关键不变量。


## 6. 激活峰值：记录 F 后存活、对应 B 后释放

不做 activation recomputation 时，每个 forward 产生一份待 backward 激活。GPipe 前后分离会堆积更多在途微批；1F1B 提前回收。真实字节还取决于层、sequence、hidden、dtype 和需要保存的算子中间量。


In [ ]:
def peak_live_activations(timeline, stages):  # 定义本节可复用的核心函数。
    live, peak = [set() for _ in range(stages)], [0] * stages  # 计算并保存当前步骤的中间状态。
    for running in timeline:  # 遍历输入元素以累积或检查结果。
        for op in running:  # 遍历输入元素以累积或检查结果。
            if op.kind == "F":  # 按当前条件选择后续控制路径。
                live[op.stage].add(op.microbatch)  # 执行当前语句以推进本节示例。
            else:  # 处理前置条件不成立的分支。
                live[op.stage].remove(op.microbatch)  # 执行当前语句以推进本节示例。
            peak[op.stage] = max(peak[op.stage], len(live[op.stage]))  # 计算并保存当前步骤的中间状态。
    return peak, live  # 返回当前分支计算出的结果。

# flush 结束不应残留激活，1F1B 在至少一个前级 stage 上降低峰值。
gpipe_peak, gpipe_live = peak_live_activations(gpipe_timeline, STAGES)  # 计算并保存当前步骤的中间状态。
onef_peak, onef_live = peak_live_activations(onef_timeline, STAGES)  # 计算并保存当前步骤的中间状态。
assert all(not state for state in gpipe_live + onef_live)  # 用受控断言验证关键不变量。
assert all(a <= b for a, b in zip(onef_peak, gpipe_peak))  # 用受控断言验证关键不变量。
assert any(a < b for a, b in zip(onef_peak, gpipe_peak))  # 用受控断言验证关键不变量。


## 7. 不等长微批的 loss：按样本或 token 数加权

最后一个微批可能更小，若把各微批 mean loss 再等权平均，会改变全局目标。正确做法是累加 loss sum 与有效元素数，或对每个 mean 乘本批计数；梯度累积也必须采用同样权重。


In [ ]:
# 用简单线性模型验证分微批的加权梯度等于完整 batch 梯度。
torch.manual_seed(166)  # 执行当前语句以推进本节示例。
features = torch.randn(10, 3)  # 计算并保存当前步骤的中间状态。
targets = torch.randn(10, 1)  # 计算并保存当前步骤的中间状态。
weight_full = torch.randn(3, 1, requires_grad=True)  # 计算并保存当前步骤的中间状态。
((features @ weight_full - targets) ** 2).mean().backward()  # 执行当前语句以推进本节示例。
full_grad = weight_full.grad.clone()  # 计算并保存当前步骤的中间状态。

weight_micro = weight_full.detach().clone().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
for sl in [slice(0, 4), slice(4, 8), slice(8, 10)]:  # 遍历输入元素以累积或检查结果。
    squared = (features[sl] @ weight_micro - targets[sl]) ** 2  # 计算并保存当前步骤的中间状态。
    (squared.sum() / len(features)).backward()  # 执行当前语句以推进本节示例。
assert torch.allclose(weight_micro.grad, full_grad, atol=1e-6)  # 用受控断言验证关键不变量。
assert weight_micro.grad.shape == weight_full.shape  # 用受控断言验证关键不变量。
assert torch.isfinite(weight_micro.grad).all()  # 用受控断言验证关键不变量。


## 8. 更新屏障与消息身份：一次 flush 只能对应一个权重版本

通信包至少绑定 run、flush、microbatch、stage、方向和 weight version。所有 B 完成且梯度归并后才能推进版本；重试要幂等去重，不能把旧激活送入新权重。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Packet:  # 定义承载本节状态与行为的数据结构。
    run_id: str  # 执行当前语句以推进本节示例。
    flush_id: int  # 执行当前语句以推进本节示例。
    microbatch: int  # 执行当前语句以推进本节示例。
    src: int  # 执行当前语句以推进本节示例。
    dst: int  # 执行当前语句以推进本节示例。
    kind: str  # 执行当前语句以推进本节示例。
    weight_version: int  # 执行当前语句以推进本节示例。

def packet_key(packet):  # 定义本节可复用的核心函数。
    raw = repr(packet).encode()  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw).hexdigest()[:20]  # 返回当前分支计算出的结果。

# 相同重试键保持一致；改变权重版本或微批后必须成为不同消息。
packet = Packet("run-a", 7, 2, 1, 2, "activation", 11)  # 计算并保存当前步骤的中间状态。
assert packet_key(packet) == packet_key(packet)  # 用受控断言验证关键不变量。
assert packet_key(packet) != packet_key(Packet("run-a", 7, 2, 1, 2, "activation", 12))  # 用受控断言验证关键不变量。
assert packet.src != packet.dst  # 用受控断言验证关键不变量。


## 面试收束：怎样把实现讲成工程答案

建议按“目标与约束 → 数据/张量合同 → 核心公式 → 正确性 oracle → 性能与安全边界 → 发布门禁”作答。Notebook 里的小张量和受控状态机只证明机制成立，不等于真实集群吞吐、真实模型质量或生产安全性。上线前还要补齐目标硬件 profiling、故障注入、分布式一致性、真实数据切片、权限审计、版本化制品和回滚演练。

可继续追问：规模扩大后哪个状态最贵？哪条等价性可作为回归测试？输入或版本变化时怎样拒绝静默错误？指标改善是否只是成本、数据污染或评测器偏差造成的？
